# Fig 5B — ATAC-profile UMAP colored by (RNA-defined) pseudotime
Same ATAC-only UMAP, colored by the RNA `dpt_pseudotime` mapped onto ATAC cells via the shared multiome barcode. The black→white line is the **RNA main trajectory drawn in chromatin space** — the running-median ATAC-UMAP position across 80 pseudotime bins, root (NE, dot) → tip (Differentiated, arrow). A smooth gradient with the trajectory tracking it means the differentiation path is **also visible in chromatin**. Spearman correlation of pseudotime with ATAC-UMAP position summarizes the gradient.

In [ ]:
from paperfig_style import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt


In [ ]:
from scipy.stats import spearmanr
df = pd.read_csv(os.path.join(CSV_DIR, 'ATAC_UMAP.csv'))
pt_map = load_obs_by_atac('dpt_pseudotime')          # {atac_cellName: pseudotime}
df['pt'] = df['cellName'].map(pt_map)
df = df[df['pt'].notna()]
print('cells with pseudotime:', len(df))
# orient by whichever UMAP axis best tracks pseudotime (just for the reported number)
rho = max(abs(spearmanr(df['pt'], df['UMAP1'])[0]), abs(spearmanr(df['pt'], df['UMAP2'])[0]))
fig, ax = plt.subplots(figsize=(3.6, 3.2))
o = np.argsort(df['pt'].values)                      # draw high pt on top
sc = ax.scatter(df['UMAP1'].values[o], df['UMAP2'].values[o], c=df['pt'].values[o],
                cmap=PSEUDOTIME_CMAP, s=1.4, linewidths=0, rasterized=True)
# main RNA trajectory drawn in chromatin space: running-median ATAC-UMAP position
# across 80 RNA-pseudotime bins, connected root(NE) -> tip(Differentiated)
u = df[['UMAP1','UMAP2']].values; ptv = df['pt'].values
NB = 80
oo = np.argsort(ptv, kind='mergesort'); tb = np.empty(len(ptv), int)
tb[oo] = (np.arange(len(ptv)) * NB) // len(ptv)
cx = np.array([np.median(u[tb==b, 0]) for b in range(NB)])
cy = np.array([np.median(u[tb==b, 1]) for b in range(NB)])
rm = lambda a, w=5: np.array([np.median(a[max(0,i-w//2):i+w//2+1]) for i in range(len(a))])
sx, sy = rm(cx), rm(cy)
ax.plot(sx, sy, color='black', lw=2.6, solid_capstyle='round', zorder=5)
ax.plot(sx, sy, color='white', lw=0.9, solid_capstyle='round', zorder=6)
ax.scatter(sx[0], sy[0], s=34, color='black', edgecolor='white', lw=0.8, zorder=7)  # root
ax.annotate('', xy=(sx[-1], sy[-1]), xytext=(sx[-4], sy[-4]),
            arrowprops=dict(arrowstyle='-|>', color='black', lw=2), zorder=7)         # direction
cb = fig.colorbar(sc, ax=ax, fraction=0.045, pad=0.02); cb.set_label('Pseudotime based on RNA profile')
cb.outline.set_linewidth(0.5)
ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel('ATAC-UMAP1'); ax.set_ylabel('ATAC-UMAP2')
for sp in ['left','bottom']: ax.spines[sp].set_visible(False)
savepanel(fig, 'Fig5B_ATAC_UMAP_pseudotime')
